In [35]:
!pip install lime
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import shap
from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore", category=FutureWarning)
torch.manual_seed(42)
np.random.seed(42)
print("Imports OK")

Defaulting to user installation because normal site-packages is not writeable
Imports OK


In [36]:
file = pd.read_pickle("dd_all_cycles.pkl")

In [37]:
pd.set_option("display.max_colwidth", None)
print(file.pv.head(10))
len(file)

0    HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0SDS-DODData
1    HBL-010RFC:RFS-DIG-101:Dwn0-Cmp1SDS-DODData
2    HBL-010RFC:RFS-DIG-201:Dwn0-Cmp0SDS-DODData
3    HBL-010RFC:RFS-DIG-201:Dwn0-Cmp1SDS-DODData
4    HBL-010RFC:RFS-DIG-301:Dwn0-Cmp0SDS-DODData
5    HBL-010RFC:RFS-DIG-301:Dwn0-Cmp1SDS-DODData
6    HBL-010RFC:RFS-DIG-401:Dwn0-Cmp0SDS-DODData
7    HBL-010RFC:RFS-DIG-401:Dwn0-Cmp1SDS-DODData
8    HBL-010RFC:RFS-DIG-101:Dwn5-Cmp0SDS-DODData
9    HBL-010RFC:RFS-DIG-101:Dwn5-Cmp1SDS-DODData
Name: pv, dtype: object


218533

In [38]:
test = []

for i in range(len(file)):
    if "HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0" in str(file.iloc[i]["pv"]):
        test.append(file.iloc[i])

test = pd.DataFrame(test)

In [30]:
print(test.pv.head(10))

0      HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0SDS-DODData
24     HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0SDS-DODData
48     HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0SDS-DODData
72     HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0SDS-DODData
96     HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0SDS-DODData
120    HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0SDS-DODData
144    HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0SDS-DODData
168    HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0SDS-DODData
192    HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0SDS-DODData
216    HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0SDS-DODData
Name: pv, dtype: object


In [51]:
test

,cycle_id,cycle_ts,pv,data_ts,data_array,trig_name,array_dimensions.binning,trig_user_tag,array_dimensions.offset,array_dimensions.reverse,...,source_archive,source_parquet,source_list_line,source_row,data_array_std,data_array_max,data_array_mean,data_array_5000_15000_std,data_array_5000_15000_mean,data_array_5000_15000_ptp
0,15643300178,2025-05-29 12:28:19.832928256+00:00,HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0SDS-DODData,2025-05-29 12:28:19.838728192+00:00,"[0.21542916, 0.21506907, 0.21506907, 0.21506907, 0.21506907, 0.21506907, 0.21506907, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21517578, 0.21517578, 0.2152958, 0.21536082, 0.21542916, 0.21557578, 0.21565406, 0.21582057, 0.21600029, 0.21629457, 0.21661837, 0.2168506, 0.21709584, 0.21762522, 0.21805611, 0.21851581, 0.21917328, 0.21988122, 0.22083634, 0.221656, 0.2227491, 0.22367771, 0.22515947, 0.22647433, 0.2281464, 0.23022422, 0.23179021, 0.23375729, 0.23617032, 0.23833708, 0.24136305, 0.24413495, 0.24743953, 0.2508844, 0.25400928, 0.25864467, 0.2620043, 0.26695982, 0.27156934, 0.27631316, 0.28173336, 0.2861769, 0.29185835, 0.29767337, 0.30361438, 0.31028625, 0.3164686, 0.32338998, 0.3304302, 0.33758184, 0.34483796, 0.35219213, 0.35963827, 0.36717084, 0.37548062, 0.3838808, 0.39236552, 0.40092942, 0.40956756, 0.419004, 0.42778242, 0.4366219, 0.4462628, 0.45521823, 0.4649771, 0.4740349, 0.48389816, 0.4945754, 0.5045383, 0.5145449, 0.5238184, 0.53467935, 0.5448028, 0.5557437, 0.5667225, 0.57694924, 0.5872052, 0.59828097, 0.6093872, 0.61972594, 0.63088626, 0.6412727, 0.65248185, ...]",HBL-010RFC:RFS-EVR-101,[0],0,[0],[False],...,/lustre/work/pdl18409/shared/data/anomaly-detection/raw/SDS_Data_20250528_2045_20250530_0416_DD_15643300300.tar_bgntn.gz,SDS_Data/20250528_2045_20250530_0416/DD/15643300300/HBL.parquet,1,64,7.689251,19.790163,5.557866,7.377371,11.243440,18.428512
24,15643300178,2025-05-29 12:28:19.832928256+00:00,HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0SDS-DODData,2025-05-29 12:28:19.838728192+00:00,"[0.21542916, 0.21506907, 0.21506907, 0.21506907, 0.21506907, 0.21506907, 0.21506907, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21517578, 0.21517578, 0.2152958, 0.21536082, 0.21542916, 0.21557578, 0.21565406, 0.21582057, 0.21600029, 0.21629457, 0.21661837, 0.2168506, 0.21709584, 0.21762522, 0.21805611, 0.21851581, 0.21917328, 0.21988122, 0.22083634, 0.221656, 0.2227491, 0.22367771, 0.22515947, 0.22647433, 0.2281464, 0.23022422, 0.23179021, 0.23375729, 0.23617032, 0.23833708, 0.24136305, 0.24413495, 0.24743953, 0.2508844, 0.25400928, 0.25864467, 0.2620043, 0.26695982, 0.27156934, 0.27631316, 0.28173336, 0.2861769, 0.29185835, 0.29767337, 0.30361438, 0.31028625, 0.3164686, 0.32338998, 0.3304302, 0.33758184, 0.34483796, 0.35219213, 0.35963827, 0.36717084, 0.37548062, 0.3838808, 0.39236552, 0.40092942, 0.40956756, 0.419004, 0.42778242, 0.4366219, 0.4462628, 0.45521823, 0.4649771, 0.4740349, 0.48389816, 0.4945754, 0.5045383, 0.5145449, 0.5238184, 0.53467935, 0.5448028, 0.5557437, 0.5667225, 0.57694924, 0.5872052, 0.59828097, 0.6093872, 0.61972594, 0.63088626, 0.6412727, 0.65248185, ...]",HBL-010RFC:RFS-EVR-101,[0],0,[0],[False],...,/lustre/work/pdl18409/shared/data/anomaly-detection/raw/SDS_Data_20250528_2045_20250530_0416_DD_15643300300_jypcu.tar,SDS_Data/20250528_2045_20250530_0416/DD/15643300300/HBL.parquet,2,64,7.689251,19.790163,5.557866,7.377371,11.243440,18.428512
48,15643300192,2025-05-29 12:28:20.832928256+00:00,HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0SDS-DODData,2025-05-29 12:28:20.838728192+00:00,"[0.2155008, 0.21506907, 0.21506907, 0.21506907, 0.21506907, 0.21506907, 0.21506907, 0.21506907, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21517578, 0.21517578, 0.21517578, 0.21517578, 0.21523413, 0.2152958, 0.2152958, 0.21542916, 0.2155008, 0.21565406, 0.21573566, 0.21590878, 0.21609509, 0.21629457, 0.21661837, 0.21685

In [39]:
file

,cycle_id,cycle_ts,pv,data_ts,data_array,trig_name,array_dimensions.binning,trig_user_tag,array_dimensions.offset,array_dimensions.reverse,...,source_archive,source_parquet,source_list_line,source_row,data_array_std,data_array_max,data_array_mean,data_array_5000_15000_std,data_array_5000_15000_mean,data_array_5000_15000_ptp
0,15643300178,2025-05-29 12:28:19.832928256+00:00,HBL-010RFC:RFS-DIG-101:Dwn0-Cmp0SDS-DODData,2025-05-29 12:28:19.838728192+00:00,"[0.21542916, 0.21506907, 0.21506907, 0.21506907, 0.21506907, 0.21506907, 0.21506907, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21512076, 0.21517578, 0.21517578, 0.2152958, 0.21536082, 0.21542916, 0.21557578, 0.21565406, 0.21582057, 0.21600029, 0.21629457, 0.21661837, 0.2168506, 0.21709584, 0.21762522, 0.21805611, 0.21851581, 0.21917328, 0.21988122, 0.22083634, 0.221656, 0.2227491, 0.22367771, 0.22515947, 0.22647433, 0.2281464, 0.23022422, 0.23179021, 0.23375729, 0.23617032, 0.23833708, 0.24136305, 0.24413495, 0.24743953, 0.2508844, 0.25400928, 0.25864467, 0.2620043, 0.26695982, 0.27156934, 0.27631316, 0.28173336, 0.2861769, 0.29185835, 0.29767337, 0.30361438, 0.31028625, 0.3164686, 0.32338998, 0.3304302, 0.33758184, 0.34483796, 0.35219213, 0.35963827, 0.36717084, 0.37548062, 0.3838808, 0.39236552, 0.40092942, 0.40956756, 0.419004, 0.42778242, 0.4366219, 0.4462628, 0.45521823, 0.4649771, 0.4740349, 0.48389816, 0.4945754, 0.5045383, 0.5145449, 0.5238184, 0.53467935, 0.5448028, 0.5557437, 0.5667225, 0.57694924, 0.5872052, 0.59828097, 0.6093872, 0.61972594, 0.63088626, 0.6412727, 0.65248185, ...]",HBL-010RFC:RFS-EVR-101,[0],0,[0],[False],...,/lustre/work/pdl18409/shared/data/anomaly-detection/raw/SDS_Data_20250528_2045_20250530_0416_DD_15643300300.tar_bgntn.gz,SDS_Data/20250528_2045_20250530_0416/DD/15643300300/HBL.parquet,1,64,7.689251,19.790163,5.557866,NaN,NaN,NaN
1,15643300178,2025-05-29 12:28:19.832928256+00:00,HBL-010RFC:RFS-DIG-101:Dwn0-Cmp1SDS-DODData,2025-05-29 12:28:19.838728192+00:00,"[15.939585, 89.99326, 89.99326, 89.99326, 44.993134, 44.993134, 44.993134, 26.563644, 26.563644, 26.563644, 26.563644, 26.563644, 26.563644, 26.563644, 26.563644, 26.563644, 26.563644, 33.68365, 33.68365, 36.865974, 38.656467, 33.68365, 32.005066, 33.68365, 34.98456, 33.68365, 35.530098, 32.90031, 32.27084, 31.753277, 32.005066, 30.648207, 30.962942, 31.21473, 29.578106, 29.291348, 28.067379, 27.47288, 27.402937, 26.94832, 26.563644, 25.899202, 25.339674, 25.12985, 24.14368, 23.56317, 23.262423, 22.807804, 22.402145, 21.877588, 21.415976, 21.143206, 20.63963, 19.982183, 19.583519, 19.226818, 18.61833, 18.247643, 17.737072, 17.191532, 16.932749, 16.471138, 15.974556, 15.582886, 15.163239, 14.771568, 14.414868, 14.079151, 13.764416, 13.435693, 13.169916, 12.883158, 12.610387, 12.211722, 11.973923, 11.757105, 11.386417, 11.1975765, 10.868853, 10.554117, 10.281347, 9.994589, 9.728812, 9.477024, 9.218242, 8.994431, 8.672701, 8.462877, 8.155137, 7.9732895, 7.686531, 7.4137607, 7.2459016, 7.008102, 6.7703023, 6.6234255, 6.4136024, 6.280714, 6.0778847, 5.8750553, ...]",HBL-010RFC:RFS-EVR-101,[0],0,[0],[False],...,/lustre/work/pdl18409/shared/data/anomaly-detection/raw/SDS_Data_20250528_2045_20250530_0416_DD_15643300300.tar_bgntn.gz,SDS_Data/20250528_2045_20250530_0416/DD/15643300300/HBL.parquet,1,65,18.397422,89.993263,-25.604455,NaN,NaN,NaN
2,15643300178,2025-05-29 12:28:19.832928256+00:00,HBL-010RFC:RFS-DIG-201:Dwn0-Cmp0SDS-DODData,2025-05-29 12:28:19.838728192+00:00,"[0.12046938, 0.12046938, 0.12046938, 0.12046938, 0.12046938, 0.12046938, 0.12046938, 0.12046938, 0.12046938, 0.12046938, 0.12046938, 0.12046938, 0.12057827, 0.12057827, 0.12057827, 0.12057827, 0.12057827, 0.12069454, 0.12069454, 0.12081817, 0.12081817, 0.120949134, 0.12108741, 0.12123297, 0.1213858, 0.12154585, 0.12188754, 0.12206912, 0.12245358, 0.12265639, 0.12308301, 0.12353738, 0.12427029, 0.12452813, 0.12534167, 0.12621433, 0.12682834, 0.12779681, 0.12882106, 0.